Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

## Setup & Imports

In [ ]:
from pathlib import Path
import requests
from tqdm.auto import tqdm
import concurrent.futures
from typing import List, Tuple

# Configuration
LOCATIONS = [
    "bamako",
    "guangdong_province",
    "guatemala_department",
    "lusaka_district",
    "nakuru",
]

QUARTERS = [
    f"{year}q{q}"
    for year in range(2020, 2026)
    for q in range(1, 5)
    if not (year == 2025 and q > 2)
][1:]  # from 2020q2 up to 2025q2

BASE_URL = "https://opendata.aiforgood.ai/building-density/locations/{location}/{yq}_cog.tif"

DEFAULT_DOWNLOAD_DIR = Path("../data/tempo_tiles")

print(f"Available locations: {', '.join(LOCATIONS)}")
print(f"Available quarters: {QUARTERS[0]} to {QUARTERS[-1]} ({len(QUARTERS)} total)")

Available locations: bamako, guangdong_province, guatemala_department, lusaka_district, nakuru
Available quarters: 2020q2 to 2025q2 (21 total)


## Helper Functions

In [3]:
def download_tile(location: str, yq: str, output_dir: Path, overwrite: bool = False) -> Tuple[bool, str]:
    """
    Download a single tile for a specific location and quarter.
    
    Args:
        location: Location identifier (e.g., 'nakuru')
        yq: Year-quarter string (e.g., '2020q2')
        output_dir: Directory to save the downloaded file
        overwrite: Whether to overwrite existing files
        
    Returns:
        Tuple of (success: bool, message: str)
    """
    location_dir = output_dir / location
    location_dir.mkdir(parents=True, exist_ok=True)
    
    output_path = location_dir / f"{yq}_cog.tif"
    
    if output_path.exists() and not overwrite:
        return True, f"Already exists: {output_path.relative_to(output_dir)}"
    
    url = BASE_URL.format(location=location, yq=yq)
    
    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(output_path, 'wb') as f:
            if total_size == 0:
                f.write(response.content)
            else:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        
        return True, f"Downloaded: {output_path.relative_to(output_dir)}"
    
    except requests.exceptions.RequestException as e:
        return False, f"Failed to download {location}/{yq}: {str(e)}"
    except Exception as e:
        return False, f"Error downloading {location}/{yq}: {str(e)}"


def download_multiple_tiles(
    locations: List[str],
    quarters: List[str],
    output_dir: Path,
    overwrite: bool = False,
    max_workers: int = 4
) -> dict:
    """
    Download multiple tiles in parallel with progress tracking.
    
    Args:
        locations: List of location identifiers
        quarters: List of year-quarter strings
        output_dir: Directory to save the downloaded files
        overwrite: Whether to overwrite existing files
        max_workers: Maximum number of parallel downloads
        
    Returns:
        Dictionary with download statistics
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    tasks = [(loc, yq) for loc in locations for yq in quarters]
    
    results = {
        'total': len(tasks),
        'success': 0,
        'already_exists': 0,
        'failed': 0,
        'failed_items': []
    }
    
    with tqdm(total=len(tasks), desc="Downloading tiles", unit="tile") as pbar:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_task = {
                executor.submit(download_tile, loc, yq, output_dir, overwrite): (loc, yq)
                for loc, yq in tasks
            }
            
            for future in concurrent.futures.as_completed(future_to_task):
                loc, yq = future_to_task[future]
                try:
                    success, message = future.result()
                    
                    if success:
                        if "Already exists" in message:
                            results['already_exists'] += 1
                        else:
                            results['success'] += 1
                    else:
                        results['failed'] += 1
                        results['failed_items'].append((loc, yq, message))
                    
                    pbar.set_postfix({
                        'success': results['success'],
                        'exists': results['already_exists'],
                        'failed': results['failed']
                    })
                    
                except Exception as e:
                    results['failed'] += 1
                    results['failed_items'].append((loc, yq, str(e)))
                
                pbar.update(1)
    
    return results


def print_download_summary(results: dict):
    """Print a summary of download results."""
    print("\n" + "="*60)
    print("DOWNLOAD SUMMARY")
    print("="*60)
    print(f"Total tiles:        {results['total']}")
    print(f"Downloaded:         {results['success']}")
    print(f"Already existed:    {results['already_exists']}")
    print(f"Failed:             {results['failed']}")
    print("="*60)
    
    if results['failed'] > 0:
        print("\nFailed downloads:")
        for loc, yq, msg in results['failed_items']:
            print(f"  - {loc}/{yq}: {msg}")

## 1. Download a Single Tile

Download one specific tile for a location and quarter.

In [ ]:
# Configuration
location = "nakuru"  # Change this to your desired location
quarter = "2024q1"   # Change this to your desired quarter
output_dir = DEFAULT_DOWNLOAD_DIR
overwrite = False    # Set to True to re-download existing files

print(f"Downloading tile for {location} - {quarter}")
print(f"Output directory: {output_dir}")

success, message = download_tile(location, quarter, output_dir, overwrite)

if success:
    print(f"✓ {message}")
else:
    print(f"✗ {message}")

## 2. Download All Tiles for a Specific Quarter

Download tiles for all locations for a given quarter.

In [ ]:
# Configuration
quarter = "2024q1"  # Change this to your desired quarter
output_dir = DEFAULT_DOWNLOAD_DIR
overwrite = False
max_workers = 4  # Number of parallel downloads

print(f"Downloading all locations for quarter: {quarter}")
print(f"Locations: {', '.join(LOCATIONS)}")
print(f"Output directory: {output_dir}")
print()

results = download_multiple_tiles(
    locations=LOCATIONS,
    quarters=[quarter],
    output_dir=output_dir,
    overwrite=overwrite,
    max_workers=max_workers
)

print_download_summary(results)

## 3. Download All Tiles for a Specific Location

Download tiles for all quarters for a given location.

In [ ]:
# Configuration
location = "nakuru"  # Change this to your desired location
output_dir = DEFAULT_DOWNLOAD_DIR
overwrite = False
max_workers = 4  # Number of parallel downloads

print(f"Downloading all quarters for location: {location}")
print(f"Quarters: {QUARTERS[0]} to {QUARTERS[-1]} ({len(QUARTERS)} total)")
print(f"Output directory: {output_dir}")
print()

results = download_multiple_tiles(
    locations=[location],
    quarters=QUARTERS,
    output_dir=output_dir,
    overwrite=overwrite,
    max_workers=max_workers
)

print_download_summary(results)

## 4. Download All Available Tiles

Download all tiles for all locations and all quarters. **Warning: This will download a ~9 GB of data!**

In [ ]:
# Configuration
output_dir = DEFAULT_DOWNLOAD_DIR
overwrite = False
max_workers = 4  # Number of parallel downloads (increase for faster downloads if bandwidth allows)

print("=" * 60)
print("DOWNLOADING ALL TEMPO TILES")
print("=" * 60)
print(f"Locations: {len(LOCATIONS)} - {', '.join(LOCATIONS)}")
print(f"Quarters: {len(QUARTERS)} - {QUARTERS[0]} to {QUARTERS[-1]}")
print(f"Total tiles: {len(LOCATIONS) * len(QUARTERS)}")
print(f"Output directory: {output_dir}")
print(f"Parallel workers: {max_workers}")
print("=" * 60)
print("\n⚠️  WARNING: This will download a large amount of data!")
print("Estimated total size: 9 GB")
print()

# Uncomment the following lines to proceed with the download
# results = download_multiple_tiles(
#     locations=LOCATIONS,
#     quarters=QUARTERS,
#     output_dir=output_dir,
#     overwrite=overwrite,
#     max_workers=max_workers
# )

print_download_summary(results)

print("To proceed, uncomment the code in this cell and run it again.")

## 5. Download Specific Subsets

Create custom download patterns by specifying exactly which locations and quarters you want.

In [ ]:
# Configuration - customize as needed
selected_locations = ["nakuru", "bamako"]  # Only these locations
selected_quarters = ["2024q1", "2024q2", "2024q3", "2024q4"]  # Only these quarters
output_dir = DEFAULT_DOWNLOAD_DIR
overwrite = False
max_workers = 4

print(f"Custom download configuration:")
print(f"  Locations: {', '.join(selected_locations)}")
print(f"  Quarters: {', '.join(selected_quarters)}")
print(f"  Total tiles: {len(selected_locations) * len(selected_quarters)}")
print(f"  Output directory: {output_dir}")
print()

results = download_multiple_tiles(
    locations=selected_locations,
    quarters=selected_quarters,
    output_dir=output_dir,
    overwrite=overwrite,
    max_workers=max_workers
)

print_download_summary(results)

## 6. Verify Downloaded Files

With the following script, you can check which files have been downloaded and verify the directory structure.

In [15]:
def verify_downloads(output_dir: Path) -> dict:
    """
    Verify which tiles have been downloaded.
    
    Args:
        output_dir: Directory containing downloaded tiles
        
    Returns:
        Dictionary with verification statistics
    """
    if not output_dir.exists():
        print(f"Directory does not exist: {output_dir}")
        return {}
    
    stats = {
        'locations': {},
        'total_files': 0,
        'total_size_mb': 0
    }
    
    for location in LOCATIONS:
        location_dir = output_dir / location
        if location_dir.exists():
            files = list(location_dir.glob("*.tif"))
            total_size = sum(f.stat().st_size for f in files)
            
            stats['locations'][location] = {
                'count': len(files),
                'size_mb': total_size / (1024 * 1024),
                'quarters': sorted([f.stem.replace('_cog', '') for f in files])
            }
            
            stats['total_files'] += len(files)
            stats['total_size_mb'] += total_size / (1024 * 1024)
    
    return stats


# Verify downloaded files
print(f"Verifying downloads in: {DEFAULT_DOWNLOAD_DIR}")
print()

stats = verify_downloads(DEFAULT_DOWNLOAD_DIR)

if not stats:
    print("No downloads found.")
else:
    print("="*60)
    print("DOWNLOAD VERIFICATION")
    print("="*60)
    print(f"Total files: {stats['total_files']}")
    print(f"Total size: {stats['total_size_mb']:.2f} MB")
    print("="*60)
    print()
    
    for location in LOCATIONS:
        if location in stats['locations']:
            loc_stats = stats['locations'][location]
            print(f"{location}:")
            print(f"  Files: {loc_stats['count']}")
            print(f"  Size: {loc_stats['size_mb']:.2f} MB")
            print(f"  Quarters: {', '.join(loc_stats['quarters'][:5])}", end="")
            if len(loc_stats['quarters']) > 5:
                print(f" ... ({len(loc_stats['quarters'])} total)")
            else:
                print()
            print()

Verifying downloads in: ../data/tempo_tiles

DOWNLOAD VERIFICATION
Total files: 105
Total size: 9205.59 MB

bamako:
  Files: 21
  Size: 57.02 MB
  Quarters: 2020q2, 2020q3, 2020q4, 2021q1, 2021q2 ... (21 total)

guangdong_province:
  Files: 21
  Size: 8840.55 MB
  Quarters: 2020q2, 2020q3, 2020q4, 2021q1, 2021q2 ... (21 total)

guatemala_department:
  Files: 21
  Size: 185.19 MB
  Quarters: 2020q2, 2020q3, 2020q4, 2021q1, 2021q2 ... (21 total)

lusaka_district:
  Files: 21
  Size: 79.12 MB
  Quarters: 2020q2, 2020q3, 2020q4, 2021q1, 2021q2 ... (21 total)

nakuru:
  Files: 21
  Size: 43.71 MB
  Quarters: 2020q2, 2020q3, 2020q4, 2021q1, 2021q2 ... (21 total)



## Tips

### Download Configuration

- **max_workers**: Increase this value (e.g., 8-16) for faster downloads if you have good bandwidth
- **overwrite**: Set to `True` to re-download files (useful if previous downloads were corrupted)
- **output_dir**: Change this to store files in a different location

### File Structure

Downloaded files are organized as:
```
data/tempo_tiles/
├── bamako/
│   ├── 2020q2_cog.tif
│   ├── 2020q3_cog.tif
│   └── ...
├── nakuru/
│   ├── 2020q2_cog.tif
│   └── ...
└── ...
```